# RDP HTTPX — Synchronous Historical Interday Summaries

This notebook demonstrates a complete end-to-end HTTP requests flow against [LSEG Data Platform (RDP)](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis) REST API using [`httpx` library](https://www.python-httpx.org/) in **synchronous** mode.

## What is Data Platform APIs?

[LSEG Data Platform](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis) (RDP APIs, also known as Delivery Platform in LSEG Real-Time) provides simple web based API access to a broad range of LSEG content.
 
For more detail regarding the Data Platform, please see the following APIs resources: 
- [Quick Start](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis/quick-start) page.
- [Tutorials](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis/tutorials) page.

## What is HTTPX?

[HTTPX](https://www.python-httpx.org/) is a full featured modern HTTP client for Python 3. It provides a set of synchronous and modern asynchronous APIs with [HTTP/2](https://httpwg.org/specs/rfc7540.html) supported. 

I am demonstrating the RDP HTTP REST APIs workflow with a share HTTPX [`httpx.Client`](https://www.python-httpx.org/advanced/clients/) object (equivalent to [`requests.Session()`](https://requests.readthedocs.io/en/latest/user/advanced/#session-objects) object in the [Requests library](https://requests.readthedocs.io/en/latest/)).

## Code Walkthrough 

### Importing Libraries

The first step is to importing the require libraries which are `os`, `time`, `httpx`, and `dotenv`.


In [2]:
import os
import time
import httpx
from IPython.display import Markdown, display
from dotenv import load_dotenv

### Configuration

API endpoint paths and the RIC universe are defined as constants. Credentials (`MACHINEID_RDP`, `PASSWORD_RDP`, `APPKEY_RDP`, `RDP_BASE_URL`) are loaded from `src/.env` via `python-dotenv` library.

Please create your own `src/.env` file using `src/.env.example` template.

In [3]:
AUTH_TOKEN_URL = "/auth/oauth2/v1/token"
AUTH_REVOKE_URL = "/auth/oauth2/v1/revoke"
HISTORICAL_INTERDAY_SUMMARIES_URL = "/data/historical-pricing/v1/views/interday-summaries/"
HISTORICAL_RICS = ["NVDA.O","AAPL.O","MSFT.O","AMZN.O","GOOG.O","AVGO.O","META.O","ORCL.N","IBM.N","PLTR.O","NFLX.O","TSLA.O","CRM.N","AMD.O","INTC.O","ARM.O","ASML.AS","CSCO.O","WMT.O","LLY.N","JPM.N","XOM.N","V.N","JNJ.N","MU.O","MA.N","COST.O","CVX.N","BAC.N","CAT.N"] # Fetched sequentially
print(f"Number of RICs to fetch sequentially: {len(HISTORICAL_RICS)}")

Number of RICs to fetch sequentially: 30


In [4]:
# reads variables from a .env file and sets them in os.environ or os.getenv method.

load_dotenv()

# Read credentials and base URL from environment variables.
machine_id = os.getenv("MACHINEID_RDP")
password = os.getenv("PASSWORD_RDP")
app_key = os.getenv("APPKEY_RDP")
base_url = os.getenv("RDP_BASE_URL")

fields = ["TRDPRC_1", "BID", "ASK"]
start_date = "2025-11-01T00:00:00Z"
end_date = "2026-02-28T23:59:59Z"

### Helper Functions

Three reusable functions used by the main execution block. All accept the shared `httpx.Client` instance so the same connection pool is reused across every call. They are all using synchronous programming model. 

| Function | Purpose |
|---|---|
| `post_authentication` | OAuth 2.0 Password Grant — returns a Bearer token |
| `post_auth_revoke` | Invalidates the session token when done |
| `get_historical_interday_summaries` | Fetches daily OHLCV data for a single RIC |

### Authentication

The Data Platform APIs entitlement check is based on OAuth 2.0 specification. The first step of an application work flow is to get a token, which will allow access to the protected resource, i.e. data REST API's.

You can find more detail from the following resources:

- [RDP APIs: Authorization - All about tokens](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis/tutorials#authorization-all-about-tokens) page.
- [RDP APIs: Authorization in Python](https://developers.lseg.com/en/api-catalog/refinitiv-data-platform/refinitiv-data-platform-apis/tutorials#authorization-in-python) page.

In [5]:
def post_authentication(machine_id, password, app_key, url, client):
    """Authenticate to RDP and return the token response as JSON."""

    # Build the OAuth 2.0 Password Grant request payload.
    # Sent as application/x-www-form-urlencoded (httpx encodes a dict automatically).
    payload = {
        "username": machine_id,           # RDP Machine-ID
        "password": password,             # RDP Password
        "grant_type": "password",         # OAuth 2.0 grant type
        "scope": "trapi",                 # Target API scope
        "takeExclusiveSignOnControl": "true",  # Revoke other active sessions
        "client_id": app_key              # RDP AppKey (acts as client_id)
    }

    # Send authentication request to the OAuth token endpoint.
    # `data=payload` sends a form body required by this endpoint.
    response = client.post(url, data=payload)
    response.raise_for_status()  # Raise for 4xx/5xx API failures.
    return response.json()

In [6]:
def post_auth_revoke(token, app_key, url, client):
    """Revoke the access token to end the session."""
    headers = {
        "Content-Type": "application/x-www-form-urlencoded"
    }

    payload = f"token={token}"
    auth = httpx.BasicAuth(username=app_key, password="")
    response = client.post(url, data=payload, headers=headers, auth=auth)
    response.raise_for_status()

Once the authentication success, the function gets the RDP Auth service response message and keeps the following RDP token information in the variables.

- **access_token**: The token used to invoke REST data API calls as described above. The application must keep this credential for further RDP APIs requests.
- **refresh_token**: Refresh token to be used for obtaining an updated access token before expiration. The application must keep this credential for access token renewal.
- **expires_in**: Access token validity time in seconds.

### Requesting RDP APIs Data

That brings us to requesting the RDP APIs data. All subsequent REST API calls use the Access Token via the `Authorization` HTTP request message header as shown below to get the data. 
- Header: 
    * Authorization = ```Bearer <RDP Access Token>```

Please notice *the space* between the ```Bearer``` and ```RDP Access Token``` values.

The application then creates a request message in a JSON message format or URL query parameter based on the interested service and sends it as an HTTP request message to the API Endpoint. Developers can get RDP APIs the API Endpoint, HTTP operations, and parameters from Data Platform's [API Playground page](https://apidocs.refinitiv.com/Apps/ApiDocs) - which is an interactive documentation site developers can access once they have a valid Data Platform account.

I am demonstrating with the `/data/historical-pricing/v1/views/interday-summaries/{universe}` endpoint. This endpoint lets developers retrieve time series pricing Interday summaries data (i.e. bar data).

In [7]:
def get_historical_interday_summaries(ric, token, url, client, interval, start, end, fields):
    """Request historical Interday summaries data for a single RIC."""
    print(f"Fetching historical interday summaries... for RIC: {ric}")

    # Bearer token authenticates the caller; Content-Type signals a JSON response is expected.
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }

    # Build the query string for the interday-summaries endpoint.
    # adjustments: apply standard corporate-action and price corrections so that
    # the returned series is comparable across the full date range.
    # fields: comma-separated list of data columns to include in the response.
    params = {
        "interval": interval,
        "start": start,
        "end": end,
        "adjustments": "exchangeCorrection,manualCorrection,CCH,CRE,RPO,RTS",
        "fields": ",".join(fields)
    }

    response_history =  client.get(f"{url}{ric}", params=params, headers=headers)

    print(f"Request URL: {response_history.url}")

    # Raise an exception for 4xx/5xx HTTP errors; lets the caller handle
    # status-specific logic (e.g. 429 rate-limit vs. 401 auth failure).
    response_history.raise_for_status()

    # Deserialise and return the JSON payload for further processing by the caller.
    return response_history.json()

### Main Execution

This notebook performs the following HTTP Operation tasks with the Data Platform REST APIs in synchronous execution model.

1. Authenticate with OAuth 2.0 Password Grant to obtain a Bearer token
2. Fetch historical interday price summaries (daily OHLCV) for a list of RICs
3. Revoke the session token on completion

While the top-level API (`httpx.get`, `httpx.post`, etc.) establishes a new connection in every single request, a share [`http.Client`](https://www.python-httpx.org/advanced/clients/) lets an application reuse underlying TCP connection when an application make HTTP requests to the same host. The `httpx.Client` instance helps an application improves performance as its reduces latency, CPU usages, and network congestion.

> **Note:** `verify=False` disables TLS certificate verification — intended for local/dev environments only (e.g. behind a ZScaler proxy). Remove it for production use.

### Open a shared `httpx.Client`

Using a `with` block ensures the connection pool is closed (and open connections flushed) when the block exits — whether it completes normally or raises an exception. All three API calls share the same underlying TCP connections, reducing overhead compared to creating a new client per request.

The code opens a single shared `httpx.Client`, then run the full workflow: **authenticate → fetch data → revoke token**.

In [9]:
with httpx.Client(
    verify=False,
    base_url=base_url,
    timeout=10.0,
    default_encoding="utf-8",
    follow_redirects=True,
) as client:
    try:
        # Authenticate and get the access token.
        auth_response = post_authentication(machine_id, password, app_key, AUTH_TOKEN_URL, client)
        access_token = auth_response["access_token"]
        print("Authentication successful! Access token obtained.\n")

        # Fetch historical interday summaries for each RIC sequentially.
        # For better performance with many RICs, consider concurrent requests (e.g. using asyncio or threading).

        display(Markdown("**Start the wall-clock timer...**"))
        start_time = time.perf_counter()
        
        for ric in HISTORICAL_RICS:
            history_data =  get_historical_interday_summaries(
                ric, access_token, HISTORICAL_INTERDAY_SUMMARIES_URL, client, interval="P1D", start=start_date, end=end_date, fields=fields
            )
            print("Historical interday summaries retrieved successfully!")
            print(f"Historical interday summaries for '{ric}': {history_data}\n\n")

        elapsed = time.perf_counter() - start_time
        elapsed_string = f"**Sending Historical-Pricing for ({len(HISTORICAL_RICS)} RICs in {elapsed:0.2f} seconds.**"
        display(Markdown(elapsed_string))
        
        time.sleep(1)  # Sleep briefly to ensure all requests are completed before revoking the token.
        print("Revoking access token...")
        post_auth_revoke(access_token, app_key, AUTH_REVOKE_URL, client)
        print("Access token revoked successfully.\n")

    except httpx.HTTPStatusError as exc:
        print(f"HTTP error occurred during HTTP Request: {exc.request.url}: {exc.response.status_code} - {exc.response.text}")
    except httpx.RequestError as exc:
        print(f"An error occurred during HTTP Request: {exc.request.url}: {exc}")
    except ValueError as exc:
        print(f"Configuration error: {exc}")
    except KeyError as exc:
        print(f"An error occurred during HTTP Request: {exc}")

Authentication successful! Access token obtained.



**Start the wall-clock timer...**

Fetching historical interday summaries... for RIC: NVDA.O
Request URL: https://api.refinitiv.com/data/historical-pricing/v1/views/interday-summaries/NVDA.O?interval=P1D&start=2025-11-01T00%3A00%3A00Z&end=2026-02-28T23%3A59%3A59Z&adjustments=exchangeCorrection%2CmanualCorrection%2CCCH%2CCRE%2CRPO%2CRTS&fields=TRDPRC_1%2CBID%2CASK
Historical interday summaries retrieved successfully!
Historical interday summaries for 'NVDA.O': [{'universe': {'ric': 'NVDA.O'}, 'interval': 'P1D', 'summaryTimestampLabel': 'endPeriod', 'adjustments': ['exchangeCorrection', 'manualCorrection', 'CCH', 'CRE', 'RTS', 'RPO'], 'defaultPricingField': 'TRDPRC_1', 'headers': [{'name': 'DATE', 'type': 'string'}, {'name': 'TRDPRC_1', 'type': 'number', 'decimalChar': '.'}, {'name': 'BID', 'type': 'number', 'decimalChar': '.'}, {'name': 'ASK', 'type': 'number', 'decimalChar': '.'}], 'data': [['2026-02-27', 177.19, 177.14, 177.15], ['2026-02-26', 184.89, 184.87, 184.89], ['2026-02-25', 195.56, 195.69, 195.88], ['2026-02-2

**Sending Historical-Pricing for (30 RICs in 2.67 seconds.**

Revoking access token...
Access token revoked successfully.



Lastly, print out the time used by all HTTP operations again.

In [10]:
print(f"sync_call_nb.ipynb executed for ({len(HISTORICAL_RICS)} RICs) in {elapsed:0.2f} seconds.")

sync_call_nb.ipynb executed for (30 RICs) in 2.67 seconds.
